# 03 — Data Cleaning & Masking Augmentation

Two jobs in this notebook:

1. **Basic cleaning**: replace the `-9999` sentinel with real `NaN` so every
   downstream function (aggregation, indices, the model itself) treats
   missingness as missingness, not as a bogus numeric value.
2. **Masking augmentation**: manufacture masked copies of train that
   structurally match test's window pattern (4-6 consecutive months, plus the
   measured S2-cloud-dropout rate within that window). This is the fix for the
   central problem identified in notebook 01 — train is fully populated, test
   never is, and a model trained only on full rows has never practiced making
   a decision from partial evidence.

Output of this notebook: `data/processed/train_augmented.csv` and
`data/processed/test_clean.csv`, ready for notebook 04's feature engineering.

In [1]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd

from src.config import (
    ALL_BANDS, S1_BANDS, S2_BANDS, MONTHS, PROCESSED_DIR,
    RANDOM_SEED,
)
from src.data_utils import (
    load_raw_train, load_raw_test, per_band_missing_rate, active_window_size,
)
from src.features import to_nan
from src.masking import build_augmented_train

pd.set_option('display.max_columns', 20)

train = load_raw_train()
test = load_raw_test()
print(train.shape, test.shape)

(1821, 146) (1030, 145)


## 1. Basic cleaning: -9999 -> NaN

In [2]:
train_nan = to_nan(train)
test_nan = to_nan(test)

# sanity: train should now have zero -9999 left, test's NaN rate should match
# what we measured in notebook 01
print("train has any -9999 left?", (train_nan.select_dtypes(include=[float, int]) == -9999).values.any())
print("test has any -9999 left?", (test_nan.select_dtypes(include=[float, int]) == -9999).values.any())

train has any -9999 left? False
test has any -9999 left? False


**Why this matters:** `-9999` is a placeholder, not a value — if it survives
into aggregation or modeling, it silently distorts means/stds and gives tree
models an artificial split point that means "masked," not anything about ponds.
Converting to real `NaN` up front means every function downstream can rely on
pandas'/numpy's native missing-value handling instead of each one having to
remember to special-case `-9999`.

## 2. Masking augmentation on train

In [3]:
N_AUGMENTS_PER_ROW = 10  # each original train row -> 10 randomly-masked variants

train_augmented = build_augmented_train(train_nan, n_augments_per_row=N_AUGMENTS_PER_ROW, seed=RANDOM_SEED)

print("original train rows:", len(train))
print("augmented train rows:", len(train_augmented))
train_augmented[['ID', 'origin_id', 'variant_id', 'window_start_month', 'window_length', 'label']].head(12)

original train rows: 1821
augmented train rows: 18210


,ID,origin_id,variant_id,window_start_month,window_length,label
0,ID_TR_NEW_XVGKFMLNRJ_v0,ID_TR_NEW_XVGKFMLNRJ,0,05,6,0
1,ID_TR_NEW_GP8KNSWVP6_v0,ID_TR_NEW_GP8KNSWVP6,0,04,4,0
2,ID_TR_NEW_87X3957MVS_v0,ID_TR_NEW_87X3957MVS,0,04,6,1
3,ID_TR_NEW_T4JMRPKHS3_v0,ID_TR_NEW_T4JMRPKHS3,0,04,5,0
4,ID_TR_NEW_2CTUQQ8KLU_v0,ID_TR_NEW_2CTUQQ8KLU,0,05,4,0
5,ID_TR_NEW_5CGQF2NXSP_v0,ID_TR_NEW_5CGQF2NXSP,0,02,5,0
6,ID_TR_NEW_W6CWRKWE8Z_v0,ID_TR_NEW_W6CWRKWE8Z,0,07,6,0
7,ID_TR_NEW_7DXZY43UUL_v0,ID_TR_NEW_7DXZY43UUL,0,04,4,0
8,ID_TR_NEW_WYYQTN7879_v0,ID_TR_NEW_WYYQTN7879,0,06,6,1
9,ID_TR_NEW_W6V6S5TLFN_v0,ID_TR_NEW_W6V6S5TLFN,0,05,6,0


**Why 10 copies per row?** Each copy uses an independently-sampled random
window and cloud-dropout pattern, so 10 copies of the same pond give the model
10 different partial views of it (different seasons, different missing-S2
months) rather than one fixed view. This is the mechanism that teaches the
model to be robust to *which* window it's handed at test time.

**Critical note carried into notebook 05:** `origin_id` identifies which
variants came from the same physical location. Cross-validation folds MUST be
grouped by `origin_id` (e.g. `GroupKFold`), never split by row. If two masked
views of the same pond end up on opposite sides of a train/validation split,
the model gets to "peek" at a near-duplicate of a validation example during
training — CV score would look better than it actually is.

## 3. Verify the augmentation actually matches test's structure

In [4]:
# Window-size distribution: augmented train vs real test
# (month_missing_matrix now recognizes both -9999 and NaN as missing, so this
# works the same whether we pass raw or already-cleaned dataframes)
aug_window = active_window_size(train_augmented, band='VH')
test_window = active_window_size(test_nan, band='VH')

print("Augmented train window-size distribution:")
print(pd.Series(aug_window).value_counts(normalize=True).sort_index())
print()
print("Real test window-size distribution:")
print(pd.Series(test_window).value_counts(normalize=True).sort_index())

Augmented train window-size distribution:
4    0.337727
5    0.335091
6    0.327183
Name: proportion, dtype: float64

Real test window-size distribution:
4    0.334951
5    0.333010
6    0.332039
Name: proportion, dtype: float64


In [5]:
# S2-given-S1 dropout rate: augmented train vs real test
def s2_dropout_rate_generic(df, s1_band='VH', s2_band='blue'):
    s1_present = df[[f'{s1_band}_{m}' for m in MONTHS]].notna().values
    s2_present = df[[f'{s2_band}_{m}' for m in MONTHS]].notna().values
    s1_on = s1_present
    s2_off_given_s1_on = (~s2_present) & s1_on
    return s2_off_given_s1_on.sum() / s1_on.sum()

print("Augmented train S2-given-S1 dropout rate:", s2_dropout_rate_generic(train_augmented))

from src.data_utils import s2_dropout_given_s1_rate
print("Real test S2-given-S1 dropout rate:", s2_dropout_given_s1_rate(test))

Augmented train S2-given-S1 dropout rate: 0.06138149640097735
Real test S2-given-S1 dropout rate: 0.062172139110161256


**Why this matters:** this is the check that actually validates the
augmentation worked as intended — not just "code ran," but "the resulting
distribution genuinely resembles what the model will face at test time." If
these numbers didn't line up, everything built on top of this augmented data
(features, model, CV) would be learning to solve the wrong problem.

## 4. Per-band missing rates: augmented train vs test (sanity check)

In [6]:
aug_missing = pd.Series({
    b: train_augmented[[f'{b}_{m}' for m in MONTHS]].isna().values.mean()
    for b in ALL_BANDS
}).sort_values(ascending=False)

test_missing = per_band_missing_rate(test)

pd.DataFrame({'augmented_train_missing_rate': aug_missing, 'test_missing_rate': test_missing})

,augmented_train_missing_rate,test_missing_rate
green,0.609734,0.609466
blue,0.609734,0.609466
re1,0.609734,0.609466
red,0.609734,0.609466
nir,0.609734,0.609466
nira,0.609734,0.609466
re2,0.609734,0.609466
re3,0.609734,0.609466
swir1,0.609734,0.609466
swir2,0.609734,0.609466


**How to read this:** these two columns should be close band-by-band. S1
bands (VH, VV) should show a slightly lower missing rate than S2 bands in both
columns, because S2 gets the extra cloud-dropout on top of the shared window
masking — if that pattern shows up in both columns similarly, augmentation is
faithful to what test actually looks like.

## 5. Save processed data

In [7]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

train_augmented.to_csv(PROCESSED_DIR / 'train_augmented.csv', index=False)
test_nan.to_csv(PROCESSED_DIR / 'test_clean.csv', index=False)

print("saved:", PROCESSED_DIR / 'train_augmented.csv', train_augmented.shape)
print("saved:", PROCESSED_DIR / 'test_clean.csv', test_nan.shape)

saved: /home/claude/project/data/processed/train_augmented.csv (18210, 150)
saved: /home/claude/project/data/processed/test_clean.csv (1030, 145)


## Summary of findings (actual results)

- `-9999` fully replaced by NaN in both train and test — confirmed zero
  remaining sentinel values.
- Train augmented from 1,821 rows into 18,210 masked variants (10 per row),
  each carrying `origin_id` (original ID, for grouped CV) and `variant_id`.
- **Window-size match:** augmented train is 33.8% / 33.5% / 32.7% across
  4/5/6-month windows; real test is 33.5% / 33.3% / 33.2%. Effectively identical.
- **S2-given-S1 dropout match:** augmented train 6.14%, real test 6.22% —
  within noise of each other.
- **Per-band missing rate match:** every band's missing rate in augmented
  train is within ~0.03 percentage points of test's (e.g. `green`: 60.97% vs
  60.95%; `VH`: 58.42% vs 58.36%). The augmentation reproduces test's
  structure very closely, band by band.

**Carried into notebook 04:** feature engineering runs on
`data/processed/train_augmented.csv` (18,210 rows) and
`data/processed/test_clean.csv` (1,030 rows), both NaN-clean and structurally
matched. Aggregates (mean/std/min/max over valid months) and the index
computations from `src/features.py` will now see genuinely comparable
missingness patterns on both sides.

**Carried into notebook 05:** `origin_id` must drive `GroupKFold` (or similar)
during cross-validation — never a plain KFold, since 10 variants of the same
physical location currently sit in the augmented set and must stay together
on one side of any split.